# 02 — Prepare canonical analysis inputs

Load the authoritative corrected stream and write the single canonical set of analysis times, surveyed receiver geometry, SLC-40 coordinates, channel metadata, and key-event annotations consumed downstream.


Load and validate authoritative corrected products, define source times, and write standardized geometry/event metadata.

In [1]:

from pathlib import Path
import json
import sys

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
MODULE_DIR = PROJECT_ROOT / "modules"
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from obspy import Stream, UTCDateTime
from pyproj import Geod

from project_config import ensure_output_dirs

PATHS = ensure_output_dirs(PROJECT_ROOT)
DERIVED_DIR = PATHS["derived"]
FIGURE_DIR = PATHS["figures"]

plt.rcParams.update({
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})


In [2]:

from obspy import read_inventory
from bchh_io import read_corrected_bchh_stream, validate_trace_units

RESPONSE_DIR = PATHS["response_correction"]
CORRECTED_PICKLE = RESPONSE_DIR / "BCHH_20160901_response_corrected.pkl"
CORRECTED_MSEED = RESPONSE_DIR / "BCHH_20160901_response_corrected.mseed"
CORRECTED_XML = RESPONSE_DIR / "BCHH_20160901_empirically_calibrated.xml"

EVENT_TIME = UTCDateTime("2016-09-01T13:07:12.080")
LOAD_START = EVENT_TIME - 180
LOAD_END = EVENT_TIME + 1800

st_corr = read_corrected_bchh_stream(
    CORRECTED_PICKLE,
    CORRECTED_MSEED,
    starttime=LOAD_START,
    endtime=LOAD_END,
    station="BCHH",
    location="00",
)
validate_trace_units(st_corr)
inventory = read_inventory(CORRECTED_XML)


Loaded corrected BCHH stream from: /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/outputs/response_correction/final_products/BCHH_20160901_response_corrected.pkl
6 Trace(s) in Stream:
FL.BCHH.00.HD1 | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples
FL.BCHH.00.HD2 | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples
FL.BCHH.00.HD3 | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples
FL.BCHH.00.HHE | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples
FL.BCHH.00.HHN | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples
FL.BCHH.00.HHZ | 2016-09-01T13:04:12.080000Z - 2016-09-01T13:37:12.080000Z | 250.0 Hz, 495001 samples
FL.BCHH.00.HD1: Pa
FL.BCHH.00.HD2: Pa
FL.BCHH.00.HD3: Pa
FL.BCHH.00.HHE: m/s
FL.BCHH.00.HHN: m/s
FL.BCHH.00.HHZ: m/s


## Event definitions

In [3]:

events = pd.DataFrame([
    {
        "event_id": "second_stage",
        "label": "Initial second-stage failure",
        "source_time": str(EVENT_TIME),
    },
    {
        "event_id": "principal_explosion",
        "label": "Principal explosion",
        "source_time": str(UTCDateTime("2016-09-01T13:07:15.750")),
    },
    {
        "event_id": "capsule_pulse_1",
        "label": "Capsule pulse 1",
        "source_time": str(UTCDateTime("2016-09-01T13:07:24.600")),
    },
    {
        "event_id": "capsule_pulse_2",
        "label": "Capsule pulse 2",
        "source_time": str(UTCDateTime("2016-09-01T13:07:25.150")),
    },
])
display(events)
events.to_csv(DERIVED_DIR / "key_events.csv", index=False)


,event_id,label,source_time
0,second_stage,Initial second-stage failure,2016-09-01T13:07:12.080000Z
1,principal_explosion,Principal explosion,2016-09-01T13:07:15.750000Z
2,capsule_pulse_1,Capsule pulse 1,2016-09-01T13:07:24.600000Z
3,capsule_pulse_2,Capsule pulse 2,2016-09-01T13:07:25.150000Z


## Canonical source and receiver geometry

Receiver coordinates come from the differential-GPS survey. Distances and azimuths are calculated from the coordinates and the adopted SLC-40 source coordinate; they are not independently hard-coded. Back azimuth is the direction from the receiver toward SLC-40, clockwise from north.


In [4]:
SLC40_LAT = 28.56195
SLC40_LON = -80.57719

surveyed_receivers = pd.DataFrame([
    {"channel": "HD1", "sensor": "infraBSU", "easting_m": 541816.30,
     "northing_m": 3160889.14, "latitude": 28.574222,
     "longitude": -80.572417},
    {"channel": "HD2", "sensor": "infraBSU", "easting_m": 541828.28,
     "northing_m": 3160851.47, "latitude": 28.573881,
     "longitude": -80.572296},
    {"channel": "HD3", "sensor": "infraBSU", "easting_m": 541802.34,
     "northing_m": 3160865.25, "latitude": 28.574006,
     "longitude": -80.572560},
    {"channel": "HHE", "sensor": "Trillium Compact", "easting_m": 541820.43,
     "northing_m": 3160866.50, "latitude": 28.5740171,
     "longitude": -80.572375},
    {"channel": "HHN", "sensor": "Trillium Compact", "easting_m": 541820.43,
     "northing_m": 3160866.50, "latitude": 28.5740171,
     "longitude": -80.572375},
    {"channel": "HHZ", "sensor": "Trillium Compact", "easting_m": 541820.43,
     "northing_m": 3160866.50, "latitude": 28.5740171,
     "longitude": -80.572375},
])

geod = Geod(ellps="WGS84")
forward_azimuth, receiver_to_source_azimuth, distance_m = geod.inv(
    np.full(len(surveyed_receivers), SLC40_LON),
    np.full(len(surveyed_receivers), SLC40_LAT),
    surveyed_receivers["longitude"].to_numpy(),
    surveyed_receivers["latitude"].to_numpy(),
)

bchh_geometry = surveyed_receivers.copy()
bchh_geometry["distance_m"] = distance_m
bchh_geometry["source_to_receiver_azimuth_deg"] = forward_azimuth % 360.0
bchh_geometry["back_azimuth_to_slc40_deg"] = (
    receiver_to_source_azimuth % 360.0
)

array_center = bchh_geometry.loc[
    bchh_geometry["channel"].eq("HHZ")
].iloc[0]

display(bchh_geometry)
bchh_geometry.to_csv(DERIVED_DIR / "bchh_geometry.csv", index=False)

stream_file = DERIVED_DIR / "bchh_corrected_analysis_window.pkl"
st_corr.write(str(stream_file), format="PICKLE")

analysis_configuration = {
    "schema_version": 1,
    "network": "FL",
    "station": "BCHH",
    "location": "00",
    "sampling_rate_hz": 250.0,
    "analysis_start_epoch_s": float(LOAD_START.timestamp),
    "analysis_end_epoch_s": float(LOAD_END.timestamp),
    "slc40": {"latitude": SLC40_LAT, "longitude": SLC40_LON},
    "array_reference": {
        "name": "BCHH seismometer",
        "latitude": float(array_center["latitude"]),
        "longitude": float(array_center["longitude"]),
        "easting_m": float(array_center["easting_m"]),
        "northing_m": float(array_center["northing_m"]),
        "distance_m": float(array_center["distance_m"]),
        "back_azimuth_to_slc40_deg": float(
            array_center["back_azimuth_to_slc40_deg"]
        ),
    },
    "infrasound_reference_channel": "HD2",
    "geometry_file": str(DERIVED_DIR / "bchh_geometry.csv"),
    "stream_file": str(stream_file),
    "key_events_file": str(DERIVED_DIR / "key_events.csv"),
}
(DERIVED_DIR / "analysis_configuration.json").write_text(
    json.dumps(analysis_configuration, indent=2) + "\n"
)

print("Wrote", stream_file)
print("Wrote", DERIVED_DIR / "analysis_configuration.json")


,channel,sensor,easting_m,northing_m,latitude,longitude,distance_m,source_to_receiver_azimuth_deg,back_azimuth_to_slc40_deg
0,HD1,infraBSU,541816.30,3160889.14,28.574222,-80.572417,1438.029446,18.949120,198.951402
1,HD2,infraBSU,541828.28,3160851.47,28.573881,-80.572296,1406.325899,19.905375,199.907715
2,HD3,infraBSU,541802.34,3160865.25,28.574006,-80.572560,1410.854982,18.727484,198.729698
3,HHE,Trillium Compact,541820.43,3160866.50,28.574017,-80.572375,1417.930794,19.404176,199.406479
4,HHN,Trillium Compact,541820.43,3160866.50,28.574017,-80.572375,1417.930794,19.404176,199.406479
5,HHZ,Trillium Compact,541820.43,3160866.50,28.574017,-80.572375,1417.930794,19.404176,199.406479


Wrote /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/outputs/derived/bchh_corrected_analysis_window.pkl
Wrote /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/outputs/derived/analysis_configuration.json



## Standardized geometry products for Figure 1 and subsequent analysis

This section uses the existing `kml_utils`, `xml_utils`, and
`figure1_utils` modules rather than duplicating their logic. The calibrated
event inventory written by Notebook 01 is preferred; the original KSC
StationXML is used only as an explicit fallback.


In [5]:

from kml_utils import read_kml_points
from xml_utils import inventory_stations_to_dataframe
from figure1_utils import load_station_sensor_dataframe
from geometry_products import write_geometry_products

KML_FILE = PROJECT_ROOT / "metadata" / "launchpads_cameras.kml"
ORIGINAL_STATIONXML = PROJECT_ROOT / "metadata" / "KSC.xml"
CALIBRATED_STATIONXML = (
    PATHS["response_correction"]
    / "BCHH_20160901_empirically_calibrated.xml"
)

GEOMETRY_STATIONXML = (
    CALIBRATED_STATIONXML
    if CALIBRATED_STATIONXML.exists()
    else ORIGINAL_STATIONXML
)
if GEOMETRY_STATIONXML == ORIGINAL_STATIONXML:
    print(
        "WARNING: calibrated event inventory was not found; "
        "using original KSC.xml for geometry."
    )

CHANNEL_TO_SENSOR = {
    "DHZ": "Seismometer",
    "DHN": "Seismometer",
    "DHE": "Seismometer",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
}
CHANNEL_TO_LABEL = {
    "DHZ": "BCHH",
    "DHN": "BCHH",
    "DHE": "BCHH",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
}

kml_points = read_kml_points(KML_FILE)
locations_df = (
    pd.DataFrame.from_dict(kml_points, orient="index")
    .rename_axis("kml_id")
    .reset_index()
)

inventory_event, channels_df, bchh_sensors_df = (
    load_station_sensor_dataframe(
        stationxml_file=GEOMETRY_STATIONXML,
        starttime=UTCDateTime("2016-09-01T00:00:00"),
        endtime=UTCDateTime("2016-09-02T00:00:00"),
        network_code="1R",
        station_code="BCHH",
        location_code="10",
        channel_to_sensor=CHANNEL_TO_SENSOR,
        channel_to_label=CHANNEL_TO_LABEL,
        source_crs="EPSG:4326",
        target_crs="EPSG:32617",
    )
)
stations_df = inventory_stations_to_dataframe(inventory_event)

geometry_paths = write_geometry_products(
    inventory=inventory_event,
    channels_df=channels_df,
    stations_df=stations_df,
    locations_df=locations_df,
    output_directory=DERIVED_DIR,
)
for name, path in geometry_paths.items():
    print(name, path)


inventory /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/outputs/derived/BCHH_20160901_event_inventory.xml
channels /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/outputs/derived/bchh_channels.csv
stations /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/outputs/derived/bchh_stations.csv
locations /Users/thompsong/Developer/KSCRocketSeismology/08_fireball_paper/outputs/derived/launchpad_camera_locations.csv
